---
title: "Archiving Quarto Posts in Zotero"
author: "Vahram Poghosyan"
date: "2026-10-09"
categories: ["Quarto", "Zotero", "Local Development Setup"]
format:
  html:
    code-fold: true
jupyter: python3
include-after-body:
  text: |
    <script type="application/javascript" src="../../javascript/light-dark.js"></script>
---

# Overview

This note covers two ways of getting posts from this Quarto site into Zotero: rendering them to **PDF** and attaching the files, or saving the live pages as **Web Page** items with snapshots via the Zotero Connector.

**Recommendation:**

- **Published posts (`posts/`)** → save as **Web Page** items with the Zotero Connector. It's one click, Zotero fills in the title/URL/access-date automatically, and the snapshot captures the fully rendered page (MathJax math, syntax-highlighted code). Zotero 7 can annotate HTML snapshots directly, so little is lost compared to a PDF.
- **Unpublished drafts (`unpublished_posts/`)** → render to **PDF** and attach the files. Publishing a draft means *moving* its directory from `unpublished_posts/` to `posts/`, which changes its URL — so a Web Page item saved against a draft URL will go stale. A PDF is a stable local artifact that survives the move (and works even if the draft was never pushed to the live site).

# Rendering a post to PDF

Quarto renders PDFs via LaTeX, so a TeX distribution is required. Quarto's docs recommend TinyTeX, installable through Quarto itself:

```bash
quarto install tinytex
```

Then render any single post to PDF by overriding the output format on the command line — no front-matter changes needed:

```bash
quarto render posts/network_and_security/ssl_the_internets_trust_protocol.ipynb --to pdf
```

Because this is a website project, the output lands in the project output directory, mirroring the source path:

```
_site/posts/network_and_security/ssl_the_internets_trust_protocol.pdf
```

To batch-render everything (zsh):

```zsh
for nb in posts/**/*.ipynb unpublished_posts/**/*.ipynb; do
  quarto render "$nb" --to pdf
done
```

## Typst alternative (no LaTeX required)

Quarto ≥ 1.4 bundles the [Typst](https://typst.app) engine, which also produces PDFs but compiles much faster and needs no TeX installation:

```bash
quarto render posts/<topic>/<post>.ipynb --to typst
```

Worth trying first if installing TinyTeX is undesirable; fall back to `--to pdf` if a post's formatting doesn't survive Typst.

## Caveats

- **Freeze cache:** `freeze: true` is set for both post directories, so rendering reuses cached execution output from `_freeze/` — notebooks are not re-executed. This is what you want for archiving.
- **HTML-only options are ignored:** `code-fold`, the custom grid layout, and the light/dark themes don't apply to PDF. Code blocks render expanded; this is fine for an archival copy.
- **`include-after-body` script:** the raw `<script>` include is HTML and is dropped from LaTeX/Typst output, so it's harmless — no front-matter edits needed.
- **Theme-aware images:** posts using the `*.light.png` / `*.dark.png` convention will embed whichever variant is referenced in the source (the `.light` one), since the JS swap doesn't exist in PDF.

# Adding the PDFs to Zotero

Zotero won't find metadata for personal blog posts, so create the items manually:

1. In Zotero, pick (or create) a collection, e.g. **"Blog Notes"**.
2. **New Item → Document** (or **Web Page**, if you also want to record the live URL).
3. Fill in *Title*, *Author*, and *Date* from the post's front matter; for published posts, put the live URL in the *URL* field.
4. Drag the rendered PDF from `_site/.../<post>.pdf` onto the item (or right-click → **Add Attachment → Attached Copy of File**). Zotero copies the file into its own storage, so the `_site/` copy can be regenerated or deleted freely.
5. Tag the item with the post's `categories` for parity with the site's own taxonomy.

> **Tip:** dragging a PDF straight into the library creates a standalone attachment; right-click it → **Create Parent Item** and edit the metadata by hand.

# Bulk import (all PDFs at once)

Zotero has **no official CLI for importing files** — the `zotero` binary only accepts launch flags — and as of Zotero 7 the local HTTP API is **read-only**, so there's no true offline scripting route either. But the per-PDF UI workflow isn't the only option. From worst to best for this site:

1. **Multi-select drag-and-drop.** Zotero accepts many PDFs in a single drop (select them all in Dolphin, drag into a collection). Zero setup, but not recursive and the items arrive as bare attachments with no metadata.
2. **The [zotero-folder-import](https://github.com/retorquere/zotero-folder-import) plugin** adds *File → Add Files from Folder…*, which recursively imports a directory tree and can mirror subfolders as collections (so `_site/posts/<topic>/` becomes a collection per topic). Good if folder structure is all the metadata you want.
3. **Generate a BibTeX file from the posts' front matter and import it once** — the best fit here, since every notebook already carries `title`, `author`, `date`, and `categories`. Zotero's importer honors the `file` field, attaching each PDF, and maps `keywords` to tags. One `File → Import…` ingests the whole garden with real metadata.
4. **The web API via [pyzotero](https://pyzotero.readthedocs.io)** — full automation (create items + upload attachments from a script), but it requires an API key, goes through zotero.org, is capped at 50 items per request, and everything must sync back down. Overkill for a local library.

## The BibTeX route

Run from the repo root after a full `quarto render` (and `quarto render --to pdf` pass):

```python
import json, glob, re, os

entries = []
for nb_path in sorted(glob.glob("posts/**/*.ipynb", recursive=True)
                    + glob.glob("unpublished_posts/**/*.ipynb", recursive=True)):
    if ".ipynb_checkpoints" in nb_path:
        continue
    nb = json.load(open(nb_path))
    src = "".join(nb["cells"][0]["source"]) if nb["cells"] else ""
    m = re.search(r"^---\n(.*?)\n---", src, re.S)
    if not m:
        continue
    meta = dict(re.findall(r"^(title|author|date|categories):\s*(.+)$", m.group(1), re.M))
    pdf = os.path.join("_site", os.path.splitext(nb_path)[0] + ".pdf")
    if not os.path.exists(pdf):
        print("no PDF, skipping:", nb_path)
        continue
    key = re.sub(r"\W+", "_", os.path.splitext(os.path.basename(nb_path))[0])
    cats = ", ".join(re.findall(r'"([^"]+)"', meta.get("categories", "")))
    # live URL only for published posts; draft URLs change on publish
    url = ("https://v-poghosyan.github.io/" + os.path.splitext(nb_path)[0] + ".html"
           if nb_path.startswith("posts/") else "")
    entries.append(f"""@online{{{key},
  title = {{{meta.get('title', key).strip('"')}}},
  author = {{{meta.get('author', '').strip('"')}}},
  date = {{{meta.get('date', '').strip('"')}}},
  url = {{{url}}},
  keywords = {{{cats}}},
  file = {{{os.path.abspath(pdf)}}}
}}""")

open("zotero_import.bib", "w").write("\n\n".join(entries) + "\n")
print(len(entries), "entries written to zotero_import.bib")
```

Then in Zotero: **File → Import… → zotero_import.bib**, and — critically — pick **"Copy files to the Zotero storage folder"**, *not* "Link to files". `_site/` is disposable build output that gets wiped on the next render; linked attachments would die with it, copies are safe.

Caveats:

- Titles containing raw LaTeX-special characters (`&`, `%`, `#`) may need escaping in the generated `.bib`.
- Re-importing the same `.bib` later creates duplicates — Zotero's import isn't an upsert. For incremental additions, regenerate the file with only the new posts (or just drag the new PDF in by hand).
- `zotero_import.bib` is a build artifact — keep it out of git.

# Adding posts as Web Pages (published posts)

1. Install the [Zotero Connector](https://www.zotero.org/download/connectors) browser extension and have the Zotero desktop app running.
2. Open the post on the live site, e.g. `https://v-poghosyan.github.io/posts/<topic>/<post>.html`.
3. Click the Connector button (it shows a web-page icon for blog posts). Zotero creates a **Web Page** item with title, URL, and access date, and attaches a **snapshot** — a self-contained archived copy of the rendered page.
4. In Zotero 7, double-click the snapshot to open it in the built-in reader, where it can be highlighted and annotated like a PDF.

Snapshots are taken from the *rendered* DOM, so math and code highlighting are preserved. The main thing to verify per post: pages depending heavily on the dark/light theme toggle or embedded iframes may snapshot in whichever theme was active in the browser.